# 05 — Silver-based fraud training baseline

This notebook is a **thin learning and review surface** over the reusable M019 pipeline. It does not implement a second trainer.

The production-shaped path is: exact Silver Delta versions → frozen strict-PIT vectors → chronological E1/E4 training → MLflow + validated manifest.

In [ ]:
import os
from pathlib import Path

from IPython.display import Markdown, display

from pit_fintech.data.paysim import resolve_project_root
from pit_fintech.data.paysim_lakehouse import find_latest_paysim_lakehouse_manifest
from pit_fintech.models.paysim_training import (
    DEFAULT_FIXED_FPR,
    DEFAULT_SEED,
    DEFAULT_TRAIN_NONFRAUD_PER_TYPE,
    feature_importance_rows,
    find_latest_training_manifest,
    load_training_manifest,
    manifest_summary_rows,
    run_paysim_silver_training,
)

PROJECT_ROOT = resolve_project_root(Path.cwd())
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
lakehouse_manifest_path = find_latest_paysim_lakehouse_manifest(ARTIFACT_ROOT)

## 1. Mental model before running anything

- **Silver transactions** contain pre-decision event fields but no fraud label or balance leakage.
- **Silver labels** are joined only as `y`, never as an input feature.
- **E1** asks what request-time fields can do without history.
- **E4** adds recipient history computed strictly from `prior_step < current_step`.
- Both use the same chronological split, model configuration and evaluation policy, so `E4 − E1` isolates the value of PIT history.

In [ ]:
# Safe default for notebook verification: never train implicitly.
RUN_TRAINING = os.getenv("PIT_NOTEBOOK_RUN_TRAINING", "0") == "1"
TRAIN_NONFRAUD_PER_TYPE = DEFAULT_TRAIN_NONFRAUD_PER_TYPE
SEED = DEFAULT_SEED
FIXED_FPR = DEFAULT_FIXED_FPR

execution_config = {
    "mode": "train-and-review" if RUN_TRAINING else "review-only",
    "lakehouse_manifest_found": lakehouse_manifest_path is not None,
    "train_nonfraud_per_type": TRAIN_NONFRAUD_PER_TYPE,
    "seed": SEED,
    "validation_fixed_fpr": FIXED_FPR,
    "component_lineage_required": True,
}
execution_config

The next cell calls exactly the same function as `make train` / `.\make.ps1 train`. Keep `RUN_TRAINING=False` for normal review and automated notebook verification.

Launch JupyterLab with `PIT_NOTEBOOK_RUN_TRAINING=1`. Do not edit and save this notebook to enable training. The component guard rejects uncommitted training/contract/lock changes, while documentation-only commits do not require rebuilding exact Silver versions.

In [ ]:
run_result = None
if not RUN_TRAINING:
    print("Review-only mode: this notebook will not build vectors or train models.")
elif lakehouse_manifest_path is None:
    display(Markdown("Run `.\\make.ps1 build-lakehouse -Dataset paysim` first."))
else:
    run_result = run_paysim_silver_training(
        lakehouse_manifest_path,
        project_root=PROJECT_ROOT,
        artifact_root=ARTIFACT_ROOT,
        train_nonfraud_sample_per_type=TRAIN_NONFRAUD_PER_TYPE,
        seed=SEED,
        fixed_fpr=FIXED_FPR,
    )
    print("new training manifest:", run_result[1])

## 2. Read the evidence contract first

Never start interpretation from PR-AUC alone. First confirm the dataset snapshot, exact Delta versions, FeatureSpec checksum, training-component fingerprint and zero future-read violations.

In [ ]:
training_manifest_path = (
    run_result[1] if run_result else find_latest_training_manifest(ARTIFACT_ROOT)
)
manifest = load_training_manifest(training_manifest_path) if training_manifest_path else None

if manifest is None:
    display(Markdown("No M019 training manifest yet. This is expected before the first clean run."))
else:
    source_versions = {
        f"{item.layer}.{item.table}": {
            "version": item.version,
            "rows": item.rows,
            "schema_checksum": item.schema_checksum,
            "logical_checksum": item.logical_checksum,
        }
        for item in manifest.source_tables
    }
    lineage = {
        "dataset_snapshot_id": manifest.dataset_snapshot_id,
        "feature_definition_version": manifest.feature_definition_version,
        "feature_contract_checksum": manifest.feature_contract_checksum,
        "code_commit": manifest.code_commit,
        "lakehouse_code_commit": manifest.application_lakehouse_code_commit,
        "lakehouse_component_fingerprint": manifest.application_lakehouse_component_fingerprint,
        "lineage_policy_version": manifest.lineage_policy_version,
        "training_component_fingerprint": manifest.training_component_fingerprint,
        "repository_dirty": manifest.repository_dirty,
        "vector_checksum": manifest.vector_checksum,
        "future_read_violations": manifest.future_read_violations,
        "source_versions": source_versions,
    }
    display(lineage)

## 3. Check the chronological population

`train` may downsample only non-fraud rows to control CPU/RAM. `validation` and `test` must remain natural-prevalence populations. Fraud rate can therefore be much higher in train than in validation/test; that is intentional and must not be mistaken for production prevalence.

In [ ]:
if manifest is not None:
    split_rows = [
        {
            "split": item.name,
            "steps": f"{item.step_min}-{item.step_max}",
            "rows": item.rows,
            "fraud_rows": item.fraud_rows,
            "fraud_rate": round(item.fraud_rate, 8),
            "prevalence": "natural" if item.natural_prevalence else "train-sampled",
        }
        for item in manifest.partitions
    ]
    display(split_rows)

## 4. Compare E1 and E4 on the untouched test period

- **PR-AUC** is primary because fraud is rare; compare it with the test fraud rate, not with 0.5.
- **ROC-AUC** measures ranking but can look optimistic under extreme imbalance.
- **Recall@fixed-FPR** says how much fraud is caught under the false-positive budget selected on validation.
- **Precision@fixed-FPR** says how many alerts at that threshold are actually fraud.
- A weak or negative `E4 − E1` is valid evidence for PaySim's sparse recipient history; do not tune it away.

In [ ]:
if manifest is not None:
    metric_rows = manifest_summary_rows(manifest)
    results = {item.experiment_id: item for item in manifest.experiments}
    comparison = {
        "E4_minus_E1_test_pr_auc": round(
            results["E4"].test_pr_auc - results["E1"].test_pr_auc,
            6,
        ),
        "E4_minus_E1_test_recall_at_fixed_fpr": round(
            results["E4"].test_recall_at_fixed_fpr - results["E1"].test_recall_at_fixed_fpr,
            6,
        ),
        "fixed_fpr_budget": manifest.fixed_fpr,
    }
    display(metric_rows)
    display(comparison)

## 5. Read feature importance as model usage, not causality

Gain importance shows which fields LightGBM used to reduce training loss. It does **not** prove that a feature causes fraud, generalizes to another dataset, or is leakage-free. Leakage safety comes from the FeatureSpec and temporal gates, not from importance.

In [ ]:
if manifest is not None:
    display(feature_importance_rows(manifest, experiment_id="E4"))

## 6. Recommended reading order after your run

1. Confirm `future_read_violations == 0`, exact source checksums match, and the training component has a recorded fingerprint.
2. Confirm Silver transaction/label versions and checksums are present.
3. Confirm split steps are train `1–520`, validation `521–631`, test `632–743`.
4. Confirm validation/test are marked `natural`.
5. Use PR-AUC as the main E1/E4 comparison.
6. Read recall, precision and observed FPR together; never quote recall without its false-positive cost.
7. Inspect E4 gain importance only after correctness and lineage pass.
8. Keep the claim boundary: this closes Sprint 1 feasibility, not Sprint 2 promotion or serving parity.

In [ ]:
if manifest is not None:
    print("Claim boundaries:")
    for statement in manifest.claim_boundary:
        print("-", statement)